# T06 — Nạp Qwen2.5-7B-Instruct 4-bit trên Tesla T4

Notebook này **không chứa logic nào**, chỉ clone repo, cài đặt và gọi script — theo mục 9 của `CLAUDE.md`.

**Trước khi chạy, trong Notebook settings bật:**

- Accelerator: **GPU T4 x2** (chỉ dùng 1 card, nhưng Kaggle chỉ cấp T4 theo cặp)
- Internet: **On** — không bật thì không clone và không tải mô hình được
- Add-ons → Secrets: thêm `HF_TOKEN` nếu muốn tải nhanh hơn, không bắt buộc
- Data: attach dataset `unicorn1209/vihallulens` (T06 chưa cần dữ liệu, nhưng T07 thì cần)

Chạy hết từ trên xuống rồi **copy toàn bộ output của ô cuối** dán vào PR.

In [ ]:
!git clone -q https://github.com/wsunicorn/vihallulens.git /kaggle/working/vihallulens
%cd /kaggle/working/vihallulens
!git log --oneline -1

In [ ]:
# Kaggle đã có sẵn torch dựng theo CUDA của image, nên KHÔNG cài lại torch.
# Chỉ cài gói của dự án ở chế độ không phụ thuộc để không kéo torch bản khác về.
!pip install -q --no-deps -e .
!pip install -q -U bitsandbytes accelerate transformers

In [ ]:
import os

# Khóa lấy từ Kaggle Secrets, không viết thẳng vào notebook.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("đã nạp HF_TOKEN từ Kaggle Secrets")
except Exception as error:
    print(f"không có HF_TOKEN, vẫn chạy được vì Qwen2.5 là mô hình mở ({error})")

# Dữ liệu chưa cần ở T06, nhưng gắn sẵn để T07 dùng lại notebook này.
if os.path.isdir("/kaggle/input/vihallulens") and not os.path.exists("data/raw"):
    os.makedirs("data", exist_ok=True)
    os.symlink("/kaggle/input/vihallulens", "data/raw")
    print("đã trỏ data/raw sang dataset")

In [ ]:
# Bảng ngân sách trên giấy (T05) — để đối chiếu ngay với số đo thật bên dưới.
!python scripts/probe_vram.py --skip-token-stats

In [ ]:
# T06 — số đo thật. Copy toàn bộ output của ô này dán vào PR.
!python scripts/probe_load_model.py